# Grid 4×4 — Realism benchmark (Homo, Slow-start, Hetero, Partial obs)

Corrected `sumo4x4`, seed 42. **RL**: 200 episodes (`*DTL.log`, test ATT). **Baselines**: single 3600-step eval (`*BRF.log` Final Travel Time). Each row is tied to a specific log path under `data/output_data/tsc/` — toggles are never mixed.

**Section order:** Homo → Slow-start → Hetero → Partial obs (pen, gauss). Partial-obs control is the homo (full-obs) runs in §1; `base_s42` is not a separate toggle.


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

REPO = Path('..').resolve()
OUT = REPO / 'data' / 'output_data' / 'tsc'
COLS = ['model', 'split', 'episode', 'travel_time', 'col5', 'reward', 'queue', 'delay', 'throughput']

EXPERIMENTS = {
    'homo': {
        'toggle': 'homo (default vType)',
        'rl': {
            'DQN': OUT / 'sumo_dqn_sumo4x4' / 'sumo4x4' / 'homo_4x4' / 'logger',
            'PressLight': OUT / 'sumo_presslight_sumo4x4' / 'sumo4x4' / 'homo_4x4' / 'logger',
            'CoLight': OUT / 'sumo_colight' / 'sumo4x4' / 'homo_4x4' / 'logger',
        },
        'baselines': {
            'MaxPressure': OUT / 'sumo_maxpressure' / 'sumo4x4' / 'baseline_homo' / 'logger',
            'FixedTime': OUT / 'sumo_fixedtime' / 'sumo4x4' / 'baseline_homo' / 'logger',
        },
    },
    'slow_start': {
        'toggle': 'slow_start (accel=1.0, tau=1.9)',
        'rl': {
            'DQN': OUT / 'sumo_dqn_slow_start' / 'sumo4x4' / 'slow_start_4x4' / 'logger',
            'PressLight': OUT / 'sumo_presslight_slow_start' / 'sumo4x4' / 'slow_start_4x4' / 'logger',
            'CoLight': OUT / 'sumo_colight_slow_start' / 'sumo4x4' / 'slow_start_4x4' / 'logger',
        },
        'baselines': {
            'MaxPressure': OUT / 'sumo_maxpressure_slow_start' / 'sumo4x4' / 'baseline_slow_start' / 'logger',
            'FixedTime': OUT / 'sumo_fixedtime_slow_start' / 'sumo4x4' / 'baseline_slow_start' / 'logger',
        },
    },
    'hetero': {
        'toggle': 'hetero (80% car / 20% truck)',
        'rl': {
            'DQN': OUT / 'sumo_dqn_hetero' / 'sumo4x4' / 'hetero_test' / 'logger',
            'PressLight': OUT / 'sumo_presslight_hetero' / 'sumo4x4' / 'hetero_test' / 'logger',
            'CoLight': OUT / 'sumo_colight_hetero' / 'sumo4x4' / 'hetero_test' / 'logger',
        },
        'baselines': {
            'MaxPressure': OUT / 'sumo_maxpressure_hetero' / 'sumo4x4' / 'baseline_hetero' / 'logger',
            'FixedTime': OUT / 'sumo_fixedtime_hetero' / 'sumo4x4' / 'baseline_hetero' / 'logger',
        },
    },
    'pobs_pen': {
        'toggle': 'partial obs — penalty p=0.8 (control = homo full obs)',
        'rl': {
            'DQN': OUT / 'sumo_dqn_pobs' / 'sumo4x4' / 'pen_s42' / 'logger',
            'PressLight': OUT / 'sumo_presslight_pobs' / 'sumo4x4' / 'pen_s42' / 'logger',
            'CoLight': OUT / 'sumo_colight_pobs' / 'sumo4x4' / 'pen_s42' / 'logger',
        },
        'baselines': {
            'MaxPressure': OUT / 'sumo_maxpressure_pobs' / 'sumo4x4' / 'pen_s42' / 'logger',
            'FixedTime': OUT / 'sumo_fixedtime_pobs' / 'sumo4x4' / 'pen_s42' / 'logger',
        },
    },
    'pobs_gauss': {
        'toggle': 'partial obs — Gaussian noise σ=2 (control = homo full obs)',
        'rl': {
            'DQN': OUT / 'sumo_dqn_pobs' / 'sumo4x4' / 'gauss_s42' / 'logger',
            'PressLight': OUT / 'sumo_presslight_pobs' / 'sumo4x4' / 'gauss_s42' / 'logger',
            'CoLight': OUT / 'sumo_colight_pobs' / 'sumo4x4' / 'gauss_s42' / 'logger',
        },
        'baselines': {
            'MaxPressure': OUT / 'sumo_maxpressure_pobs' / 'sumo4x4' / 'gauss_s42' / 'logger',
            'FixedTime': OUT / 'sumo_fixedtime_pobs' / 'sumo4x4' / 'gauss_s42' / 'logger',
        },
    },
}

def load_dtl(logger_dir: Path):
    """Pick the DTL log with the highest episode index (complete 200-ep run)."""
    best, best_ep, best_path = None, -1, None
    for p in sorted(logger_dir.glob('*_DTL.log')):
        df = pd.read_csv(p, sep='\t', header=None, names=COLS)
        mx = int(df['episode'].max())
        if mx > best_ep:
            best, best_ep, best_path = df, mx, p
    if best is None:
        raise FileNotFoundError(f'No DTL logs in {logger_dir}')
    return best, best_path

def parse_brf_att(path: Path) -> float:
    text = path.read_text()
    m = re.search(r'Final Travel Time is ([0-9.]+)', text)
    if not m:
        raise ValueError(f'No Final Travel Time in {path}')
    return float(m.group(1))

def split_curves(df):
    train = df[df.split == 'TRAIN'].reset_index(drop=True)
    test = df[df.split == 'TEST'].reset_index(drop=True)
    return train, test

def load_baseline(toggle: str, method: str):
    spec = EXPERIMENTS[toggle]['baselines'][method]
    if isinstance(spec, dict) and 'att' in spec:
        return spec['att'], spec['log']
    brf = sorted(spec.glob('*_BRF.log'))[-1]
    return parse_brf_att(brf), str(brf.relative_to(REPO))

def load_toggle_rl(toggle: str):
    out = {}
    for name, path in EXPERIMENTS[toggle]['rl'].items():
        df, log_path = load_dtl(path)
        train, test = split_curves(df)
        out[name] = {'df': df, 'train': train, 'test': test, 'log': log_path}
        print(f"{name:12s} ep_max={int(df.episode.max()):3d}  best_test={test.travel_time.min():.1f}s  final={test.travel_time.iloc[-1]:.1f}s  ({log_path.name})")
    return out

def plot_rl_curves(rl_data, title: str):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
    colors = {'train': '#4C72B0', 'test': '#C44E52'}
    for ax, name in zip(axes, rl_data):
        tr, te = rl_data[name]['train'], rl_data[name]['test']
        ax.plot(tr.index, tr.travel_time, color=colors['train'], linestyle='--', label='train')
        ax.plot(te.index, te.travel_time, color=colors['test'], label='test')
        ax.set_title(name)
        ax.set_xlabel('episode')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('ATT (s)')
    axes[-1].legend()
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

def make_table(rl_data, toggle: str):
    rows = []
    for name in rl_data:
        te = rl_data[name]['test']
        rows.append({
            'method': name,
            'best_test_att': round(te.travel_time.min(), 1),
            'final_test_att': round(te.travel_time.iloc[-1], 1),
            'log': rl_data[name]['log'].name,
        })
    for m in ['MaxPressure', 'FixedTime']:
        att, src = load_baseline(toggle, m)
        rows.append({
            'method': m,
            'best_test_att': round(att, 1),
            'final_test_att': round(att, 1),
            'log': Path(src).name if '/' in str(src) else src,
        })
    return pd.DataFrame(rows).sort_values('final_test_att')


## 1. Homo 4×4


In [ ]:
homo_rl = load_toggle_rl('homo')
for m in ['MaxPressure', 'FixedTime']:
    att, src = load_baseline('homo', m)
    print(f"{m:12s} baseline ATT={att:.1f}s  log={src}")


In [ ]:
plot_rl_curves(homo_rl, 'Homo 4×4 — RL learning curves (train dashed, test solid)')


In [ ]:
homo_table = make_table(homo_rl, "homo")
homo_table


## 2. Slow-start 4×4


In [ ]:
slow_rl = load_toggle_rl('slow_start')
for m in ['MaxPressure', 'FixedTime']:
    att, src = load_baseline('slow_start', m)
    print(f"{m:12s} baseline ATT={att:.1f}s  log={src}")


In [ ]:
plot_rl_curves(slow_rl, 'Slow-start 4×4 — RL learning curves')


In [ ]:
slow_table = make_table(slow_rl, "slow_start")
slow_table


## 3. Hetero 4×4


In [ ]:
hetero_rl = load_toggle_rl('hetero')
for m in ['MaxPressure', 'FixedTime']:
    att, src = load_baseline('hetero', m)
    print(f"{m:12s} baseline ATT={att:.1f}s  log={src}")


In [ ]:
plot_rl_curves(hetero_rl, 'Hetero 4×4 — RL learning curves')


In [ ]:
hetero_table = make_table(hetero_rl, "hetero")
hetero_table


## 4. Partial obs — penalty p=0.8


In [ ]:
pobs_pen_rl = load_toggle_rl('pobs_pen')
for m in ['MaxPressure', 'FixedTime']:
    att, src = load_baseline('pobs_pen', m)
    print(f"{m:12s} baseline ATT={att:.1f}s  log={src}")


In [ ]:
plot_rl_curves(pobs_pen_rl, 'Partial obs (pen p=0.8) — RL learning curves')


In [ ]:
pobs_pen_table = make_table(pobs_pen_rl, "pobs_pen")
pobs_pen_table


## 5. Partial obs — Gaussian σ=2


In [ ]:
pobs_gauss_rl = load_toggle_rl('pobs_gauss')
for m in ['MaxPressure', 'FixedTime']:
    att, src = load_baseline('pobs_gauss', m)
    print(f"{m:12s} baseline ATT={att:.1f}s  log={src}")


In [ ]:
plot_rl_curves(pobs_gauss_rl, 'Partial obs (gauss σ=2) — RL learning curves')


In [ ]:
pobs_gauss_table = make_table(pobs_gauss_rl, "pobs_gauss")
pobs_gauss_table


## 6. Cross-toggle ranking (final test / eval ATT, lower is better)


In [ ]:
rank_rows = []
rl_by_toggle = {
    'homo': homo_rl,
    'slow_start': slow_rl,
    'hetero': hetero_rl,
    'pobs_pen': pobs_pen_rl,
    'pobs_gauss': pobs_gauss_rl,
}
for toggle, data in rl_by_toggle.items():
    for method in ['DQN', 'PressLight', 'CoLight']:
        te = data[method]['test']
        rank_rows.append({
            'toggle': toggle,
            'method': method,
            'type': 'RL (200 ep)',
            'final_att': round(te.travel_time.iloc[-1], 1),
            'best_att': round(te.travel_time.min(), 1),
            'log': data[method]['log'].name,
        })
    for m in ['MaxPressure', 'FixedTime']:
        att, src = load_baseline(toggle, m)
        rank_rows.append({
            'toggle': toggle,
            'method': m,
            'type': 'baseline (3600 steps)',
            'final_att': round(att, 1),
            'best_att': round(att, 1),
            'log': Path(src).name if '/' in str(src) else src,
        })

ranking = pd.DataFrame(rank_rows).sort_values(['toggle', 'final_att']).reset_index(drop=True)
ranking['rank_in_toggle'] = ranking.groupby('toggle')['final_att'].rank(method='min').astype(int)
ranking


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
toggle_order = ['homo', 'slow_start', 'hetero', 'pobs_pen', 'pobs_gauss']
toggle_labels = ['Homo', 'Slow-start', 'Hetero', 'PObs pen', 'PObs gauss']
methods = ['DQN', 'PressLight', 'CoLight', 'MaxPressure', 'FixedTime']
palette = {'DQN': '#4C72B0', 'PressLight': '#55A868', 'CoLight': '#C44E52',
           'MaxPressure': '#8172B3', 'FixedTime': '#CCB974'}
x = range(len(toggle_order))
width = 0.15
for i, method in enumerate(methods):
    vals = [ranking[(ranking.toggle == t) & (ranking.method == method)]['final_att'].values[0] for t in toggle_order]
    ax.bar([xi + (i - 2) * width for xi in x], vals, width, label=method, color=palette[method])
ax.set_xticks(list(x))
ax.set_xticklabels(toggle_labels, rotation=15, ha='right')
ax.set_ylabel('Final ATT (s)')
ax.set_title('Sumo 4×4 seed 42 — realism toggles (lower is better)')
ax.legend(loc='upper left', ncol=2, fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## Summary

- **Homo** baselines: MaxPressure **173.26 s**, FixedTime **219.14 s** (`baseline_homo` BRF logs).
- **Slow-start**: baselines MP **186.0 s**, FT **243.7 s**; RL Jul-14 logs (`slow_start_4x4`).
- **Hetero**: baselines MP **189.35 s**, FT **240.45 s**; RL Jul-14 `hetero_test` logs (ignore Jul-04/06 outliers).
- **Partial obs**: compare pen / gauss to homo control in §1; `base_s42` = full obs (same as homo).
- All numbers parsed from committed `*DTL.log` / `*BRF.log` paths in `EXPERIMENTS` — no git fallback.
